# Đồ án nhóm – Thực hành Khai thác dữ liệu
## Đề tài 08: Khai thác luật đặc trưng của nấm ăn được và nấm độc bằng Frequent Itemset (Apriori & FP-Growth)
**Nhóm 11** – Lê Hồng Nhật · Dương Gia Kiệt · Huỳnh Ngọc Trâm · GVHD: Đinh Nguyễn Trọng Nghĩa
**Tệp:** `Nhom11_Code.ipynb`  ·  **Bản chất bài toán:** *khai thác luật kết hợp* (không có nhãn huấn luyện/dự đoán; nhãn `class` chỉ là một item trong transaction và được dùng làm vế phải để diễn giải luật).

### Hướng dẫn chạy (Run All)
1. Cài thư viện: `pip install -r requirements.txt` (cell 0.1 cũng tự cài `mlxtend`, `openpyxl` nếu máy chưa có). Phiên bản đã dùng được in ra ở cell 0.1.
2. Đặt notebook **cùng cấp** với thư mục `data/` chứa `mushrooms.csv`. Nếu không có file, cell 1.1 tự tải dữ liệu gốc từ UCI về `data/` (cần Internet).
3. Chọn **Restart & Run All**. Không cần sửa đường dẫn/biến. Mọi bảng và hình được ghi vào thư mục `results/` (`results/tables`, `results/figures`, `results/data`).
4. Thời gian chạy: từ vài phút đến khoảng 10–20 phút (phụ thuộc máy và phiên bản `mlxtend`; trên một số môi trường FP-Growth chậm hơn Apriori rất nhiều ở tập này). Các số đếm (tập phổ biến, số luật, support/confidence/lift) là **xác định**; chỉ có **thời gian chạy** thay đổi theo máy.

### Đối chiếu notebook ↔ báo cáo Word
| Mục notebook | Nội dung | Chương báo cáo |
|---|---|---|
| 1 – 2 | Đọc dữ liệu, đánh giá chất lượng, tiền xử lý | Chương 2, 3 |
| 3 | EDA gắn với câu hỏi khai thác (Chi-square, odor) | Chương 2 |
| 3.4 – 3.5 | Transaction và ma trận one-hot | Chương 3 |
| 4 – 5 | Apriori vs FP-Growth, lưới `minsup × minconf`, thời gian | Chương 4, 5 |
| 6 | Lọc luật, luật tiêu biểu | Chương 5 (5.6), 6 |
| 7 – 8 | Tổng hợp số liệu, kết luận, hạn chế | Chương 8 |

> ⚠️ Kết quả chỉ phục vụ học tập. **Không dùng các luật để quyết định ăn nấm ngoài tự nhiên.**
> **Nguồn dữ liệu:** Schlimmer, J. (1987). *Mushroom Data Set*, UCI Machine Learning Repository (bản đăng lại trên Kaggle). **Thuật toán:** Agrawal & Srikant (1994) – Apriori; Han, Pei & Yin (2000) – FP-Growth; thư viện `mlxtend` (Raschka, 2018).

## 0. Thiết lập môi trường

In [ ]:
# 0.1 Cài (nếu thiếu) và import thư viện; in phiên bản đã dùng
import sys, subprocess, importlib

def _ensure(module, pip_name=None):
    # Tự cài thư viện còn thiếu để Run All chạy được trên môi trường sạch
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])

_ensure("mlxtend")
_ensure("openpyxl")

import os, time, warnings, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import scipy, matplotlib, mlxtend
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

warnings.filterwarnings("ignore")            # tránh cảnh báo lặp làm phình output và nhiễu số đo thời gian
try:
    display                                  # có sẵn trong Jupyter/Colab
except NameError:
    display = print

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_colwidth", 90, "display.width", 180)

print("Phiên bản thư viện đã dùng:")
for name, ver in [("python", sys.version.split()[0]), ("pandas", pd.__version__), ("numpy", np.__version__),
                  ("scipy", scipy.__version__), ("matplotlib", matplotlib.__version__),
                  ("mlxtend", mlxtend.__version__)]:
    print(f"  {name:11s} {ver}")

In [ ]:
# 0.2 Đường dẫn (tương đối) và thư mục kết quả
DATA_DIR = Path("data")
RES_DIR  = Path("results")
TAB_DIR, FIG_DIR, DAT_DIR = RES_DIR / "tables", RES_DIR / "figures", RES_DIR / "data"
for d in (DATA_DIR, TAB_DIR, FIG_DIR, DAT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Thư mục làm việc:", Path.cwd().name, "| kết quả ghi vào:", RES_DIR)

## 1. Tổng quan bài toán và dữ liệu
**Câu hỏi khai thác:** những tổ hợp đặc điểm hình thái nào của nấm thường xuất hiện cùng nhãn *edible* (`class=e`) hoặc *poisonous* (`class=p`)? Hai thuật toán Apriori và FP-Growth có cho cùng kết quả không và khác nhau thế nào về thời gian?
Mỗi dòng dữ liệu là **một cây nấm** → mỗi dòng là **một transaction** gồm các item `thuộc_tính=giá_trị`.

In [ ]:
# 1.1 Đọc dữ liệu thô (đường dẫn tương đối; tự tải từ UCI nếu thiếu)
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
COLS = ["class", "cap-shape", "cap-surface", "cap-color", "bruises", "odor", "gill-attachment", "gill-spacing",
        "gill-size", "gill-color", "stalk-shape", "stalk-root", "stalk-surface-above-ring",
        "stalk-surface-below-ring", "stalk-color-above-ring", "stalk-color-below-ring", "veil-type",
        "veil-color", "ring-number", "ring-type", "spore-print-color", "population", "habitat"]

def load_raw():
    for p in (DATA_DIR / "mushrooms.csv", Path("mushrooms.csv")):
        if p.exists():
            print(f"-> Đọc dữ liệu từ: {p}")
            return pd.read_csv(p)
    print("-> Không thấy mushrooms.csv, tải dữ liệu gốc từ UCI ...")
    try:
        urllib.request.urlretrieve(UCI_URL, DATA_DIR / "agaricus-lepiota.data")
    except Exception as e:
        raise FileNotFoundError("Không tải được dữ liệu. Hãy đặt 'mushrooms.csv' vào thư mục 'data/' "
                                "(cạnh notebook) rồi Run All lại.") from e
    d = pd.read_csv(DATA_DIR / "agaricus-lepiota.data", header=None, names=COLS)
    d.to_csv(DATA_DIR / "mushrooms.csv", index=False)
    return d

raw = load_raw()
print("Kích thước dữ liệu thô:", raw.shape, "(dòng, cột)")
assert raw.shape == (8124, 23), "Kích thước khác dữ liệu Mushroom chuẩn (8124 x 23)"
display(raw.head())

In [ ]:
# 1.2 Ý nghĩa thuộc tính (dùng để diễn giải luật ở mục 6)
DESC = {
 "class": "nhãn: e = ăn được (edible), p = độc (poisonous)", "cap-shape": "hình dạng mũ nấm",
 "cap-surface": "bề mặt mũ", "cap-color": "màu mũ", "bruises": "có vết bầm hay không (t/f)",
 "odor": "mùi", "gill-attachment": "cách phiến mang gắn vào cuống", "gill-spacing": "khoảng cách giữa các phiến mang",
 "gill-size": "kích thước phiến mang (b = rộng, n = hẹp)", "gill-color": "màu phiến mang",
 "stalk-shape": "hình dạng cuống (e = nở dần, t = thon dần)", "stalk-root": "dạng gốc cuống ('?' = thiếu)",
 "stalk-surface-above-ring": "bề mặt cuống phía trên vòng", "stalk-surface-below-ring": "bề mặt cuống phía dưới vòng",
 "stalk-color-above-ring": "màu cuống phía trên vòng", "stalk-color-below-ring": "màu cuống phía dưới vòng",
 "veil-type": "kiểu màng bao (p = partial)", "veil-color": "màu màng bao", "ring-number": "số vòng trên cuống",
 "ring-type": "kiểu vòng", "spore-print-color": "màu dấu bào tử", "population": "mật độ quần thể",
 "habitat": "môi trường sống"}
info = pd.DataFrame({"Thuộc tính": raw.columns, "Số giá trị khác nhau": [raw[c].nunique() for c in raw.columns],
                     "Ý nghĩa": [DESC.get(c, "") for c in raw.columns]})
display(info)

## 2. Tiền xử lý và chuẩn bị dữ liệu
Các bước: (a) đánh giá chất lượng dữ liệu (thiếu, trùng, hằng số, giá trị lạ) → (b) quyết định xử lý và giải thích → (c) lưu dữ liệu sạch. Mọi con số bên dưới được **tính từ dữ liệu**, không gõ tay.

In [ ]:
# 2.1 Đánh giá chất lượng dữ liệu
n_rows, n_cols = raw.shape
miss_by_col = (raw == "?").sum()
miss_by_col = miss_by_col[miss_by_col > 0]
const_cols = [c for c in raw.columns if raw[c].nunique() == 1]
one_char = bool(raw.apply(lambda s: s.astype(str).str.len().eq(1)).all().all())
class_ok = set(raw["class"].unique()) <= {"e", "p"}

quality = pd.Series({
    "Số dòng": n_rows, "Số cột": n_cols,
    "Ô NaN (null thật)": int(raw.isna().sum().sum()),
    "Ô '?' (thiếu được mã hóa)": int((raw == "?").sum().sum()),
    "Dòng trùng lặp": int(raw.duplicated().sum()),
    "Cột hằng số": ", ".join(const_cols) if const_cols else "không có",
    "Mọi ô đều là 1 ký tự (đúng mã UCI)": one_char,
    "Nhãn class chỉ gồm {e, p}": class_ok})
print(quality.to_string())
print("\nCột có giá trị '?':")
print(pd.DataFrame({"Số ô '?'": miss_by_col, "Tỷ lệ (%)": (miss_by_col / n_rows * 100).round(2)}).to_string())

In [ ]:
# 2.2 Quyết định xử lý + giải thích
df = raw.copy()

# (i) Loại cột hằng số: support = 100% nên chỉ sinh luật vô nghĩa và làm phình số tập phổ biến
df = df.drop(columns=const_cols)

# (ii) '?' ở stalk-root: GIỮ NGUYÊN như một giá trị độc lập (item 'stalk-root=?'),
#      vì xóa dòng sẽ mất ~1/3 dữ liệu, còn điền mode làm sai lệch thông tin. Bước này không dùng nhãn -> không rò rỉ.
n_missing = int((df == "?").sum().sum())

# (iii) Dữ liệu 100% phân loại: không có ngoại lệ số, không cần chuẩn hóa/scale (Apriori/FP-Growth dựa trên đếm)

# (iv) Loại dòng trùng lặp (nếu có)
n_dup = int(df.duplicated().sum())
if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Cột đã loại (hằng số): {const_cols}")
print(f"Số ô '?' giữ nguyên như một danh mục: {n_missing}")
print(f"Số dòng trùng lặp đã loại: {n_dup}")
print(f"Kích thước sau tiền xử lý: {df.shape[0]} dòng x {df.shape[1]} cột (gồm cột class)")

df_steps = pd.DataFrame({
    "STT": [1, 2, 3, 4, 5],
    "Nội dung kiểm tra & tiền xử lý": [
        "Kiểm tra giá trị thiếu (Missing Values)", "Kiểm tra trùng lặp dòng (Duplicates)",
        "Kiểm tra giá trị bất thường / không hợp lệ", "Kiểm tra thuộc tính hằng số (độ biến thiên)",
        "Chuẩn hóa cấu trúc dữ liệu"],
    "Kết quả khảo sát": [
        "; ".join(f"'{c}' có {int(v)} ô '?' ({v / n_rows * 100:.1f}%)" for c, v in miss_by_col.items()) or "Không có",
        f"{n_dup} dòng trùng lặp",
        "Mọi ô là 1 ký tự và nhãn class thuộc {e, p}" if (one_char and class_ok) else "Có giá trị ngoài chuẩn UCI",
        f"Cột hằng số: {', '.join(const_cols) if const_cols else 'không có'}",
        f"Kết quả: {df.shape[0]} dòng, {df.shape[1]} cột"],
    "Quyết định xử lý": [
        "Giữ '?' như một danh mục độc lập", "Loại bỏ dòng trùng lặp nếu có",
        "Dữ liệu thuần phân loại, không cần lọc ngoại lai số học",
        "Loại bỏ cột hằng số để tránh luật vô nghĩa", "Dùng dữ liệu sạch cho bước tạo transaction"]})
display(df_steps)

In [ ]:
# 2.3 Lưu dữ liệu sạch và bảng mô tả tiền xử lý (Excel) - các bước sau dùng trực tiếp biến df trong bộ nhớ
XLSX_CLEAN = DAT_DIR / "Mushroom_Preprocessed_Dataset_Cleaned.xlsx"
with pd.ExcelWriter(XLSX_CLEAN, engine="openpyxl") as writer:
    df_steps.to_excel(writer, sheet_name="1. Đánh giá & Tiền xử lý", index=False)
    df.to_excel(writer, sheet_name="3. Dữ liệu đã xử lý (Cleaned)", index=False)
df_steps.to_csv(TAB_DIR / "01_tien_xu_ly.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", XLSX_CLEAN)

## 3. EDA gắn với câu hỏi khai thác và biểu diễn transaction
Chỉ thực hiện các thống kê phục vụ câu hỏi: nhãn phân bố thế nào, thuộc tính nào liên hệ mạnh với nhãn, và odor phân tách nhãn ra sao. Đây là bước **khám phá**, không phải kết luận nhân quả.

In [ ]:
# 3.1 Phân phối lớp (Hình 2.1)
class_counts = df["class"].value_counts()
class_pct = df["class"].value_counts(normalize=True) * 100
display(pd.DataFrame({"Số mẫu": class_counts, "Tỷ lệ (%)": class_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["edible (e)", "poisonous (p)"], [class_counts.get("e", 0), class_counts.get("p", 0)],
              color=["#70AD47", "#C00000"])
ax.bar_label(bars)
ax.set_title("Phân phối nhãn class"); ax.set_ylabel("Số mẫu")
fig.tight_layout(); fig.savefig(FIG_DIR / "hinh_2_1_phan_phoi_class.png", dpi=150); plt.show()

In [ ]:
# 3.2 Chi-square giữa từng thuộc tính và class (Hình 2.2)
chi2_rows = {}
for col in df.columns:
    if col == "class":
        continue
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(df[col], df["class"]))
    chi2_rows[col] = {"Chi2": chi2, "p-value": p, "dof": dof}
chi2_df = pd.DataFrame.from_dict(chi2_rows, orient="index").sort_values("Chi2", ascending=False)
chi2_df.to_csv(TAB_DIR / "02_chi2_thuoc_tinh_vs_class.csv", encoding="utf-8-sig")
print("Top 10 thuộc tính liên hệ mạnh nhất với class (Chi-square):")
display(chi2_df.head(10).round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
chi2_df.head(10)["Chi2"].plot(kind="barh", ax=ax, color="#2E75B6")
ax.invert_yaxis(); ax.set_xlabel("Thống kê Chi-square"); ax.set_ylabel("Thuộc tính")
ax.set_title("Top 10 thuộc tính liên hệ mạnh với class")
fig.tight_layout(); fig.savefig(FIG_DIR / "hinh_2_2_chi2_top10.png", dpi=150); plt.show()
print("Lưu ý: Chi-square lớn chỉ cho biết có liên hệ thống kê trong tập này; không chứng minh quan hệ nhân quả.")

In [ ]:
# 3.3 Thuộc tính 'odor' (mùi) so với class
odor_table = pd.crosstab(df["odor"], df["class"])
odor_pct = odor_table.div(odor_table.sum(axis=1), axis=0) * 100
display(pd.concat([odor_table.add_prefix("Số mẫu class="), odor_pct.round(1).add_prefix("% class=")], axis=1))
odor_table.to_csv(TAB_DIR / "03_odor_vs_class.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 4))
odor_table.plot(kind="bar", stacked=True, ax=ax, color={"e": "#70AD47", "p": "#C00000"})
ax.set_title("Phân phối odor theo class"); ax.set_xlabel("odor"); ax.set_ylabel("Số mẫu")
plt.setp(ax.get_xticklabels(), rotation=0)
fig.tight_layout(); fig.savefig(FIG_DIR / "hinh_2_3_odor_vs_class.png", dpi=150); plt.show()

# Nhận xét tính từ số liệu (không gõ tay)
pure = odor_pct[odor_pct.max(axis=1) == 100]
print("Các giá trị odor chỉ xuất hiện ở MỘT lớp (100%):",
      {o: ("e" if r["e"] == 100 else "p") for o, r in pure.iterrows()})
print(f"odor='n' (không mùi): {odor_pct.loc['n', 'e']:.1f}% là edible -> mạnh nhưng KHÔNG tuyệt đối "
      f"({int(odor_table.loc['n', 'p'])} mẫu độc cũng không mùi).")

In [ ]:
# 3.4 Tạo transaction: mỗi dòng (1 cây nấm) -> 1 giao dịch gồm các item 'thuộc_tính=giá_trị'
transactions = [[f"{c}={v}" for c, v in zip(df.columns, row)] for row in df.itertuples(index=False)]
assert all(len(t) == df.shape[1] for t in transactions), "Mỗi transaction phải có đúng 1 item cho mỗi cột"
print(f"Số giao dịch: {len(transactions)} | số item mỗi giao dịch: {len(transactions[0])}")
print("Ví dụ giao dịch đầu:", transactions[0])
print("Ví dụ giao dịch hai:", transactions[1])
with open(DAT_DIR / "transactions_list.pkl", "wb") as f:
    pickle.dump(transactions, f)

In [ ]:
# 3.5 Mã hóa one-hot (ma trận giao dịch x item) - đầu vào chung của Apriori và FP-Growth
te = TransactionEncoder()
X = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
density = X.values.sum() / X.size * 100
print(f"Ma trận one-hot: {X.shape[0]} giao dịch x {X.shape[1]} item | mật độ giá trị 1: {density:.2f}%")
display(X.iloc[:3, :8])
X.to_csv(DAT_DIR / "transactions_onehot.csv", index=False)

## 4. Phương pháp khai thác dữ liệu
**Hai thuật toán (cùng đầu vào `X`, cùng ngưỡng, cùng thư viện `mlxtend`):**
- **Apriori:** sinh tập ứng viên theo từng mức kích thước, cắt tỉa bằng tính chất *mọi tập con của tập phổ biến đều phổ biến*; đếm support bằng cách quét dữ liệu.
- **FP-Growth:** nén dữ liệu vào cây FP-Tree rồi khai thác đệ quy, không sinh ứng viên.

**Độ đo luật:** `support(A→C) = P(A∪C)`; `confidence = P(C|A)`; `lift = confidence / P(C)` (lift > 1: A và C xuất hiện cùng nhau nhiều hơn mức ngẫu nhiên).

**Lựa chọn tham số (có cơ sở):**
- `minsup ∈ {0.3, 0.4, 0.5}`: mỗi transaction có 22 item nên dữ liệu **dày**; hạ `minsup` thấp hơn làm số tập phổ biến tăng rất nhanh (tốn thời gian/RAM). Ngưỡng ≥ 0.3 đủ để có các luật phổ biến mà vẫn chạy được trên máy cá nhân.
- `minconf ∈ {0.6, 0.7, 0.8}`: từ mức tin cậy vừa phải đến cao. `minconf` chỉ tác động bước **sinh luật**, không ảnh hưởng tập phổ biến.
- Cấu hình dùng để diễn giải luật: **`minsup = 0.3`, `minconf = 0.7`** (đủ nhiều luật hướng tới `class` để chọn lọc).

**Quy trình so sánh công bằng:** chạy *warm-up* trước; mỗi cấu hình lặp `N_REPEAT = 3` lần và lấy **trung vị**; thời gian tổng = khai thác tập phổ biến + sinh luật; đồng thời ghi riêng thời gian khai thác thuần; kiểm tra Apriori và FP-Growth cho **cùng tập phổ biến và cùng bộ luật**.

In [ ]:
# 4.1 Tham số thực nghiệm và hàm chạy thuật toán
MINSUP_LIST  = [0.3, 0.4, 0.5]
MINCONF_LIST = [0.6, 0.7, 0.8]
N_REPEAT     = 3                       # lặp mỗi cấu hình, lấy trung vị để giảm nhiễu thời gian
SEL_MINSUP, SEL_MINCONF = 0.3, 0.7     # cấu hình dùng để diễn giải luật
ALGOS = {"Apriori": apriori, "FP-Growth": fpgrowth}

def run_once(algo, minsup, minconf):
    # Một lần chạy: khai thác tập phổ biến + sinh luật. Trả về (fi, rules, t_mining, t_total)
    t0 = time.perf_counter()
    fi = algo(X, min_support=minsup, use_colnames=True)
    t_mining = time.perf_counter() - t0
    rules = association_rules(fi, metric="confidence", min_threshold=minconf)
    t_total = time.perf_counter() - t0
    return fi, rules, t_mining, t_total

def rule_keys(rules):
    return set(zip(rules["antecedents"], rules["consequents"]))

def itemset_keys(fi):
    return set(fi["itemsets"])

for fn in ALGOS.values():              # warm-up: loại chi phí khởi tạo lần đầu khỏi số đo
    fn(X, min_support=0.5, use_colnames=True)
print("Sẵn sàng:", list(ALGOS), "| minsup:", MINSUP_LIST, "| minconf:", MINCONF_LIST, "| lặp:", N_REPEAT)

## 5. Thực nghiệm và đánh giá: so sánh thuật toán và khảo sát tham số

In [ ]:
# 5.1 Chạy lưới 3 minsup x 3 minconf cho cả hai thuật toán
rows_ap, rows_fp, rows_cmp = [], [], []
selected = {}                          # lưu kết quả của cấu hình được chọn để diễn giải (mục 6)

for minsup in MINSUP_LIST:
    for minconf in MINCONF_LIST:
        res = {}
        for name, algo in ALGOS.items():
            t_min, t_tot = [], []
            for _ in range(N_REPEAT):
                fi, rules, tm, tt = run_once(algo, minsup, minconf)
                t_min.append(tm); t_tot.append(tt)
            res[name] = dict(fi=fi, rules=rules, t_mining=float(np.median(t_min)), t_total=float(np.median(t_tot)))
        ap, fp = res["Apriori"], res["FP-Growth"]

        if (minsup, minconf) == (SEL_MINSUP, SEL_MINCONF):
            selected = {k: (v["fi"], v["rules"]) for k, v in res.items()}

        rows_ap.append({"Minsup": minsup, "Minconf": minconf, "Số tập phổ biến": len(ap["fi"]),
                        "Tổng số luật": len(ap["rules"]), "Thời gian (giây)": round(ap["t_total"], 4)})
        rows_fp.append({"Minsup": minsup, "Minconf": minconf, "Số tập phổ biến": len(fp["fi"]),
                        "Tổng số luật": len(fp["rules"]), "Thời gian (giây)": round(fp["t_total"], 4)})
        same_fi = itemset_keys(ap["fi"]) == itemset_keys(fp["fi"])
        same_rules = rule_keys(ap["rules"]) == rule_keys(fp["rules"])
        speedup = ap["t_total"] / fp["t_total"] if fp["t_total"] > 0 else np.nan
        rows_cmp.append({
            "Minsup": minsup, "Minconf": minconf,
            "Apriori - Số tập PB": len(ap["fi"]), "FP-Growth - Số tập PB": len(fp["fi"]),
            "Apriori - Số luật": len(ap["rules"]), "FP-Growth - Số luật": len(fp["rules"]),
            "Cùng tập phổ biến?": "Có" if same_fi else "KHÔNG", "Cùng bộ luật?": "Có" if same_rules else "KHÔNG",
            "Apriori - TG mining (s)": round(ap["t_mining"], 4), "FP-Growth - TG mining (s)": round(fp["t_mining"], 4),
            "Apriori - TG tổng (s)": round(ap["t_total"], 4), "FP-Growth - TG tổng (s)": round(fp["t_total"], 4),
            "Speedup (Apriori/FP)": round(speedup, 2),
            "Nhanh hơn": "FP-Growth" if fp["t_total"] < ap["t_total"] else "Apriori"})
        print(f"minsup={minsup} minconf={minconf} | FI {len(ap['fi'])}/{len(fp['fi'])} | "
              f"Rules {len(ap['rules'])}/{len(fp['rules'])} | cùng FI={same_fi}, cùng luật={same_rules} | "
              f"{ap['t_total']:.3f}s vs {fp['t_total']:.3f}s")

df_apriori, df_fpgrowth, df_compare = pd.DataFrame(rows_ap), pd.DataFrame(rows_fp), pd.DataFrame(rows_cmp)
df_apriori.to_csv(TAB_DIR / "04_apriori.csv", index=False, encoding="utf-8-sig")
df_fpgrowth.to_csv(TAB_DIR / "05_fpgrowth.csv", index=False, encoding="utf-8-sig")
df_compare.to_csv(TAB_DIR / "06_so_sanh_apriori_fpgrowth.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 5.2 Ba bảng kết quả và kết luận tự động từ số liệu
print("=== BẢNG 1: APRIORI ===");                 display(df_apriori)
print("\n=== BẢNG 2: FP-GROWTH ===");              display(df_fpgrowth)
print("\n=== BẢNG 3: SO SÁNH TRỰC TIẾP ===");      display(df_compare)

all_same = bool(df_compare["Cùng tập phổ biến?"].eq("Có").all() and df_compare["Cùng bộ luật?"].eq("Có").all())
n_cfg = len(df_compare)
n_fp_faster = int((df_compare["Nhanh hơn"] == "FP-Growth").sum())
print("\n=== KẾT LUẬN TỰ ĐỘNG ===")
print(f"- Tập phổ biến và bộ luật giống nhau ở cả {n_cfg} cấu hình:", "CÓ" if all_same else "KHÔNG - cần kiểm tra lại")
print(f"- FP-Growth nhanh hơn ở {n_fp_faster}/{n_cfg} cấu hình; Apriori nhanh hơn ở {n_cfg - n_fp_faster}/{n_cfg}.")
print(f"- Thời gian tổng trung bình: Apriori {df_compare['Apriori - TG tổng (s)'].mean():.3f}s, "
      f"FP-Growth {df_compare['FP-Growth - TG tổng (s)'].mean():.3f}s.")
print("  (Thời gian phụ thuộc máy, phiên bản mlxtend và mật độ dữ liệu; chỉ kết luận cho tập này + môi trường này.)")

In [ ]:
# 5.3 Biểu đồ so sánh (Hình 5.1): thời gian, số luật, số tập phổ biến theo minsup
def ve_bieu_do(dfc, out_png):
    g = dfc.groupby("Minsup").mean(numeric_only=True)
    x, w = np.arange(len(g)), 0.36
    labels = [str(m) for m in g.index]
    C_AP, C_FP = "#2E75B6", "#ED7D31"
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))

    def cap_bar(a, col_ap, col_fp, title, ylab):
        b1 = a.bar(x - w / 2, g[col_ap], w, label="Apriori", color=C_AP)
        b2 = a.bar(x + w / 2, g[col_fp], w, label="FP-Growth", color=C_FP)
        a.bar_label(b1, fmt="%.3f", fontsize=8); a.bar_label(b2, fmt="%.3f", fontsize=8)
        a.set_title(title); a.set_ylabel(ylab); a.set_xticks(x); a.set_xticklabels(labels)

    cap_bar(ax[0, 0], "Apriori - TG tổng (s)", "FP-Growth - TG tổng (s)",
            "Thời gian tổng (khai thác + sinh luật) theo minsup", "Giây (TB 3 mức minconf)")
    cap_bar(ax[0, 1], "Apriori - TG mining (s)", "FP-Growth - TG mining (s)",
            "Thời gian khai thác thuần theo minsup", "Giây (TB 3 mức minconf)")
    for mc, d in dfc.groupby("Minconf"):
        ax[1, 0].plot(d["Minsup"], d["Apriori - Số luật"], marker="o", label=f"minconf = {mc}")
    ax[1, 0].set_yscale("log"); ax[1, 0].set_xticks(sorted(dfc["Minsup"].unique()))
    ax[1, 0].set_title("Số luật theo minsup (Apriori = FP-Growth)"); ax[1, 0].set_ylabel("Số luật (trục log)")
    fi_n = dfc.groupby("Minsup")["Apriori - Số tập PB"].first()
    b = ax[1, 1].bar([str(m) for m in fi_n.index], fi_n.values, color="#70AD47")
    ax[1, 1].bar_label(b, fmt="%d")
    ax[1, 1].set_title("Số tập phổ biến theo minsup (Apriori = FP-Growth)"); ax[1, 1].set_ylabel("Số tập phổ biến")
    for a in ax.ravel():
        a.set_xlabel("Minsup"); a.grid(alpha=0.3)
        if a.get_legend_handles_labels()[0]:
            a.legend(fontsize=8)
    fig.suptitle("So sánh Apriori và FP-Growth - tập Mushroom", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_png, dpi=150)
    plt.show()
    print("=> Đã lưu biểu đồ:", out_png)

FIG_COMPARE = FIG_DIR / "hinh_5_1_so_sanh_Apriori_FPGrowth.png"
ve_bieu_do(df_compare, FIG_COMPARE)

## 6. Lọc luật, chọn luật tiêu biểu và diễn giải
Chỉ diễn giải các luật có vế phải chứa `class` (đúng câu hỏi khai thác). Các bước lọc: (B1) bỏ luật `lift ≈ 1`; (B2) bỏ luật **dư thừa** (có luật tổng quát hơn với confidence không thấp hơn); (B3) bỏ luật "đảo chiều" trùng tập item; (B4) chọn luật tiêu biểu.

In [ ]:
# 6.1 Luật mục tiêu (vế phải chứa class) tại minsup=0.3, minconf=0.7
def has_class(items):
    return any(str(i).startswith("class=") for i in items)

fi_sel, rules_all = selected["Apriori"]
rules_class = rules_all[rules_all["consequents"].apply(has_class)].copy()
n_edible = int(rules_class["consequents"].apply(lambda s: "class=e" in s).sum())
n_poison = int(rules_class["consequents"].apply(lambda s: "class=p" in s).sum())
print(f"Cấu hình chọn: minsup={SEL_MINSUP}, minconf={SEL_MINCONF}")
print(f"Tổng số luật: {len(rules_all)} | luật có class ở vế phải: {len(rules_class)} "
      f"(edible: {n_edible}, poisonous: {n_poison})")

In [ ]:
# 6.2 Các bước lọc luật (B1-B3), áp dụng cho toàn bộ luật tại cấu hình chọn
LIFT_LOW, LIFT_HIGH = 0.9, 1.1          # lift trong khoảng này coi là ~1 -> loại
def fmt(s):
    return " & ".join(sorted(s))

rules = rules_all.copy()
n0 = len(rules)

# B1. Bỏ luật lift ≈ 1 (vế trái và vế phải gần như độc lập)
rules = rules[(rules["lift"] < LIFT_LOW) | (rules["lift"] > LIFT_HIGH)].copy()
n1 = len(rules)

# B2. Bỏ luật dư thừa: X -> Y bị loại nếu có luật tổng quát hơn X' -> Y (X' ⊂ X) với confidence >= luật dài hơn
rules["k"] = rules["antecedents"].apply(len)
rules["rule_text"] = rules["antecedents"].apply(fmt) + " -> " + rules["consequents"].apply(fmt)
rules = rules.sort_values(["k", "rule_text"], kind="mergesort")         # tie-break cố định để kết quả lặp lại được
best, keep = {}, []
for idx, r in rules.iterrows():
    ante, cons, conf = r["antecedents"], r["consequents"], r["confidence"]
    redundant = any(a < ante and c >= conf - 1e-12 for a, c in best.get(cons, []))
    if not redundant:
        keep.append(idx)
        best.setdefault(cons, []).append((ante, conf))
rules = rules.loc[keep].copy()
n2 = len(rules)

# B3. Bỏ luật "đảo chiều" trùng tập item: ưu tiên chiều có class ở vế phải, rồi confidence và lift cao
rules["itemset"] = rules.apply(lambda r: frozenset(r["antecedents"] | r["consequents"]), axis=1)
rules["cls_rhs"] = rules["consequents"].apply(has_class)
rules = (rules.sort_values(["cls_rhs", "confidence", "lift", "rule_text"],
                           ascending=[False, False, False, True], kind="mergesort")
              .drop_duplicates("itemset", keep="first"))
n3 = len(rules)

df_filter = pd.DataFrame({"Bước lọc": ["Ban đầu", "Sau bỏ lift≈1", "Sau bỏ dư thừa", "Sau bỏ trùng tập item"],
                          "Số luật": [n0, n1, n2, n3]})
display(df_filter)
df_filter.to_csv(TAB_DIR / "07_thong_ke_loc_luat.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 6.3 Chọn luật tiêu biểu: 5 luật cho mỗi nhãn + 6 luật khác (mỗi thuộc tính vế phải một luật để đa dạng)
N_CLASS_PER_SIDE, N_OTHER = 5, 6
rules["score"] = rules["confidence"] * rules["lift"]
only_one = rules["consequents"].apply(len) == 1
pick = []
for cls in ["class=e", "class=p"]:
    m = only_one & rules["consequents"].apply(lambda s, c=cls: c in s)
    pick.append(rules[m].sort_values(["score", "support", "rule_text"], ascending=[False, False, True],
                                     kind="mergesort").head(N_CLASS_PER_SIDE))
oth = rules[~rules["cls_rhs"] & only_one].sort_values(["score", "rule_text"], ascending=[False, True], kind="mergesort")
oth = oth.assign(attr=oth["consequents"].apply(lambda s: next(iter(s)).split("=")[0]))
pick.append(oth.drop_duplicates("attr").head(N_OTHER))

sel = pd.concat(pick)
df_select = pd.DataFrame({
    "Luật (A -> C)": sel["antecedents"].apply(fmt) + "  ->  " + sel["consequents"].apply(fmt),
    "Support": sel["support"].round(4), "Confidence": sel["confidence"].round(4), "Lift": sel["lift"].round(3),
    "Nhóm": ["Edible" if "class=e" in c else "Poisonous" if "class=p" in c else "Khác" for c in sel["consequents"]]
}).reset_index(drop=True)
df_select.index += 1
print(f"=== {len(df_select)} LUẬT TIÊU BIỂU (minsup={SEL_MINSUP}, minconf={SEL_MINCONF}) ===")
display(df_select)
df_select.to_csv(TAB_DIR / "08_luat_tieu_bieu.csv", index_label="STT", encoding="utf-8-sig")

In [ ]:
# 6.4 Diễn giải một luật tiêu biểu bằng ngôn ngữ dễ hiểu (sinh tự động từ số liệu)
top_e = df_select[df_select["Nhóm"] == "Edible"].iloc[0]
top_p = df_select[df_select["Nhóm"] == "Poisonous"].iloc[0]
for r in (top_e, top_p):
    print(f"Luật: {r['Luật (A -> C)']}")
    print(f"  - support {r['Support']:.2%}: khoảng {r['Support']:.1%} số mẫu chứa đồng thời cả hai vế.")
    print(f"  - confidence {r['Confidence']:.2%}: trong các mẫu thỏa vế trái, {r['Confidence']:.1%} thuộc nhãn ở vế phải.")
    print(f"  - lift {r['Lift']:.3f}: cao hơn mức kỳ vọng nếu độc lập ({'có' if r['Lift'] > 1 else 'không'} liên hệ dương).")
print("\nLưu ý: đây là quan hệ đồng xuất hiện trong tập dữ liệu này, KHÔNG phải quan hệ nhân quả và không dùng để chọn nấm ăn.")

In [ ]:
# 6.5 Xuất toàn bộ kết quả ra một file Excel + hình
XLSX_OUT = RES_DIR / "Nhom11_KetQua_Apriori_FPGrowth.xlsx"
def rules_to_text(frame):
    t = frame.copy()
    for c in ["antecedents", "consequents"]:
        t[c] = t[c].apply(fmt)
    return t.drop(columns=[c for c in ["itemset"] if c in t.columns])

with pd.ExcelWriter(XLSX_OUT, engine="openpyxl") as w:
    df_apriori.to_excel(w, sheet_name="1. Apriori", index=False)
    df_fpgrowth.to_excel(w, sheet_name="2. FP-Growth", index=False)
    df_compare.to_excel(w, sheet_name="3. So sánh trực tiếp", index=False)
    rules_to_text(rules_class).to_excel(w, sheet_name="4. Luật mục tiêu class", index=False)
    df_filter.to_excel(w, sheet_name="5. Thống kê lọc", index=False)
    df_select.to_excel(w, sheet_name="6. Luật tiêu biểu", index_label="STT")
    try:                                                    # chèn hình nếu có Pillow; không bắt buộc
        from openpyxl.drawing.image import Image as XLImage
        w.book.create_sheet("7. Biểu đồ").add_image(XLImage(str(FIG_COMPARE)), "A1")
    except Exception as e:
        print("(Bỏ qua chèn hình vào Excel:", type(e).__name__, ")")
print("Đã xuất:", XLSX_OUT)

## 7. Tổng hợp số liệu chính (dùng để đối chiếu với báo cáo Word và video)

In [ ]:
# 7.1 Bảng số liệu chính - mọi con số trong Word/video phải khớp bảng này
best_rule = df_select.iloc[0]
final = pd.Series({
    "Số mẫu / số cột sau tiền xử lý": f"{df.shape[0]} / {df.shape[1]}",
    "Ma trận one-hot (giao dịch x item)": f"{X.shape[0]} x {X.shape[1]}",
    "Mật độ ma trận one-hot (%)": round(density, 2),
    "Thuộc tính có Chi-square cao nhất": f"{chi2_df.index[0]} ({chi2_df['Chi2'].iloc[0]:.2f})",
    "Số tập phổ biến (minsup 0.3 / 0.4 / 0.5)": " / ".join(str(int(df_apriori[df_apriori['Minsup'] == m]['Số tập phổ biến'].iloc[0]))
                                                           for m in MINSUP_LIST),
    "Apriori = FP-Growth ở mọi cấu hình": "Có" if all_same else "KHÔNG",
    "TG tổng trung bình Apriori (s)": round(df_compare["Apriori - TG tổng (s)"].mean(), 3),
    "TG tổng trung bình FP-Growth (s)": round(df_compare["FP-Growth - TG tổng (s)"].mean(), 3),
    f"Số luật tại minsup={SEL_MINSUP}, minconf={SEL_MINCONF}": len(rules_all),
    "Số luật có class ở vế phải": len(rules_class),
    "Số luật sau lọc (lift, dư thừa, trùng tập)": n3,
    "Luật tiêu biểu số 1": f"{best_rule['Luật (A -> C)']} (sup {best_rule['Support']}, conf {best_rule['Confidence']}, lift {best_rule['Lift']})",
})
display(final.to_frame("Giá trị"))
final.to_frame("Giá trị").to_csv(TAB_DIR / "09_tong_hop_so_lieu.csv", encoding="utf-8-sig")
print("Các file trong results/:", sorted(str(p.relative_to(RES_DIR)) for p in RES_DIR.rglob("*") if p.is_file()))

## 8. Kết luận, hạn chế và hướng phát triển
**Kết quả chính** (số liệu cụ thể xem bảng ở mục 7 và các bảng ở mục 5–6):
- Apriori và FP-Growth cho **cùng** tập phổ biến và **cùng** bộ luật ở mọi cấu hình; hai thuật toán chỉ khác nhau về **thời gian chạy**. Thời gian phụ thuộc môi trường/phiên bản `mlxtend` và đặc điểm dữ liệu (8.124 giao dịch, 118 item, mật độ ~18,6%), nên không khái quát hóa cho dữ liệu khác.
- `minsup` quyết định số tập phổ biến (giảm mạnh khi tăng `minsup`); `minconf` chỉ lọc ở bước sinh luật.
- `odor` là thuộc tính liên hệ mạnh nhất với `class` theo Chi-square; các luật tiêu biểu có confidence và lift cao khi vế phải là `class=e` hoặc `class=p`.

**Hạn chế và trường hợp có thể gây sai lệch:**
- Luật chỉ phản ánh **tập dữ liệu này**, không đại diện mọi loài nấm; confidence cao **không** là bằng chứng nhân quả hay an toàn.
- Mỗi giá trị `?` ở `stalk-root` được giữ như một danh mục riêng; cách xử lý khác (điền/xóa) có thể cho luật khác.
- Chưa kiểm chứng luật trên dữ liệu ngoài (hold-out) và chưa so sánh nhiều thư viện/cài đặt khác nhau.

**Hướng phát triển:** chia train/test để kiểm tra độ ổn định của luật; thử thêm độ đo (leverage, conviction); mở rộng so sánh với Eclat hoặc cài đặt khác; hoàn thiện sản phẩm demo tra cứu luật theo đặc điểm nhập vào.